In [ ]:
import concurrent.futures
import http.client

# Suppress logging temporarily to prevent astropy
# from repeatedly printing out warning notices related to alternate WCSs
import logging
import time
import urllib.error

import astropy.units as u
import numpy as np
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.table import Table
from astropy.visualization import quantity_support
from astropy.wcs import WCS
from astroquery.ipac.irsa import Irsa

logging.getLogger("astropy").setLevel(logging.ERROR)

# The time it takes to read SPHEREx files can exceed
# astropy's default timeout limit. Increase it.
from astropy.utils.data import conf

conf.remote_timeout = 120

# Let matplotlib recognize astropy Quantities and auto-label axes with their units.
quantity_support()

<astropy.visualization.units.quantity_support.<locals>.MplQuantityConverterFormatted at 0x7fec60934c20>

In [2]:
# Choose a position.
coord = SkyCoord(ra=210.80227 * u.degree, dec=54.34895 * u.degree, frame="icrs")

# Choose a cutout size.
size = 0.1 * u.degree

# Choose the bandpass of interest.
bandpass = "SPHEREx-D2"

# Choose an output filename root for the output MEF file.
output_filename = "spherex_cutouts_mef.fits"

In [3]:
# Define a query that will search the appropriate SPHEREx metadata tables
# for spectral images that cover the chosen coordinates and match the
# specified bandpass. Return the cutout data access URL and the time of observation.
# Sort by observation time.
query = f"""
SELECT
    'https://irsa.ipac.caltech.edu/' || a.uri || '?center={coord.ra.deg},{coord.dec.deg}d&size={size.value}' AS uri,
    p.time_bounds_lower
FROM spherex.artifact a
JOIN spherex.plane p ON a.planeid = p.planeid
WHERE 1 = CONTAINS(POINT('ICRS', {coord.ra.deg}, {coord.dec.deg}), p.poly)
        AND p.energy_bandpassname = '{bandpass}'
ORDER BY p.time_bounds_lower
"""

# Execute the query and return as an astropy Table.
t1 = time.time()
results = Irsa.query_tap(query)
print(f"Time to do TAP query: {time.time() - t1:2.2f} seconds.")
print(f"Number of images found: {len(results)}")

Time to do TAP query: 4.24 seconds.
Number of images found: 84


In [4]:
results

<DALResultsTable length=84>
                                                                                      uri                                                                                        ...
                                                                                                                                                                                 ...
                                                                                     object                                                                                      ...
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- ...
https://irsa.ipac.caltech.edu/ibe/data/spherex/qr2/level2/2025W19_2B/l2b-v20-2025-247/2/level2_2025W19_2B_0346_1D2_spx_l2b-v20-2025-247.fits?center=210.80227,54.34895d&size=0.1 ...
https://irsa.ipac.caltech.edu/ibe/data/spherex/qr2/level2/2025W19_2

In [7]:
results.to_table().to_pandas()

,uri,time_bounds_lower
0,https://irsa.ipac.caltech.edu/ibe/data/spherex...,60805.711548
1,https://irsa.ipac.caltech.edu/ibe/data/spherex...,60805.713040
2,https://irsa.ipac.caltech.edu/ibe/data/spherex...,60805.714532
3,https://irsa.ipac.caltech.edu/ibe/data/spherex...,60805.716025
4,https://irsa.ipac.caltech.edu/ibe/data/spherex...,60805.915383
...,...,...
79,https://irsa.ipac.caltech.edu/ibe/data/spherex...,61210.792447
80,https://irsa.ipac.caltech.edu/ibe/data/spherex...,61210.928366
81,https://irsa.ipac.caltech.edu/ibe/data/spherex...,61213.848301
82,https://irsa.ipac.caltech.edu/ibe/data/spherex...,61214.594837


In [ ]:
def process_cutout(row: Table, coord: SkyCoord, cache: bool = True) -> None:
    """
    Downloads the cutouts given in a row of the table including all SPHEREx images
    overlapping with a position.

    Parameters:
    ===========

    row : astropy.table row
        Row of a table that will be changed in place by this function. The table
        is created by the SQL TAP query.
    coord : astropy.coordinates.SkyCoord
        Coordinates (same as used for the TAP query).
    cache : bool
        If set to `True`, the output of cached and the cutout processing will run faster
        next time. Turn this feature off by setting ``cache=False``.
    """

    with fits.open(row["uri"], cache=cache) as hdulist:
        # There are seven HDUs:
        # 0 contains minimal metadata in the header and no data.
        # 1 through 6 are: IMAGE, FLAGS, VARIANCE, ZODI, PSF, WCS-WAVE
        header = hdulist[1].header

        # Compute pixel coordinates corresponding to cutout position.
        spatial_wcs = WCS(header)
        x, y = spatial_wcs.world_to_pixel(coord)

        # Compute wavelength at cutout position.
        spectral_wcs = WCS(header, fobj=hdulist, key="W")
        spectral_wcs.sip = None
        wavelength, bandwidth = spectral_wcs.pixel_to_world(x, y)
        row["central_wavelength"] = wavelength.to(u.micrometer).value

        # The FITS primary header carries the processing version (VERSION),
        # which isn't repeated in the extension headers. Keep it so it isn't silently
        # dropped when we build the combined MEF below.
        row["version"] = hdulist[0].header.get("VERSION", "")

        # Collect the HDUs for this cutout and append the row's cutout_index to the EXTNAME.
        hdus = []
        for hdu in hdulist[1:]:  # skip the primary header
            hdu.header["EXTNAME"] = f"{hdu.header['EXTNAME']}{row['cutout_index']}"

            # Copy so the data is available after the file is closed
            hdus.append(hdu.copy())

        row["hdus"] = hdus

In [ ]:
def process_cutout_with_error_handling(
    row: Table,
    coord: SkyCoord,
    cache: bool = True,
) -> None:
    """
    Call `process_cutout` while catching transient read errors.

    Parameters:
    ===========

    row : astropy.table row
        Row of a table that will be changed in place by this function. The table
        is created by the SQL TAP query.
    coord : astropy.coordinates.SkyCoord
        Coordinates (same as used for the TAP query).
    cache : bool
        If set to `True`, the output of cached and the cutout processing will run faster
        next time. Turn this feature off by setting ``cache=False``.
    """
    try:
        process_cutout(row, coord, cache=cache)
    except (
        TimeoutError,
        urllib.error.HTTPError,
        http.client.IncompleteRead,
        http.client.RemoteDisconnected,
    ):
        # Transient read errors. Skip this cutout.
        row["hdus"] = None

## Download the cutouts

### Serial

In [13]:
results_table_serial = results.to_table()
results_table_serial["cutout_index"] = range(1, len(results_table_serial) + 1)
results_table_serial["central_wavelength"] = np.full(len(results_table_serial), np.nan)
results_table_serial["version"] = np.full(len(results_table_serial), "", dtype="U32")
results_table_serial["hdus"] = np.full(len(results_table_serial), None)

t1 = time.perf_counter()
for row in results_table_serial:
    t0 = time.perf_counter()
    process_cutout_with_error_handling(row, coord, cache=False)
    elapsed_time = time.perf_counter() - t0
    print(f"Processed cutout {row['cutout_index']} in {(elapsed_time):2.2f} seconds.")

print(f"Time to create cutouts in serial mode: {(time.perf_counter() - t1) / 60:2.2f} minutes.")  # noqa: E501

# Drop rows that failed to download.
failed_cutouts_serial = results_table_serial[results_table_serial["hdus"] is None]
print("Failed to get the following cutouts:\n", failed_cutouts_serial["uri"])
results_table_serial = results_table_serial[results_table_serial["hdus"] is not None]

Processed cutout 1 in 5.97 seconds.
Processed cutout 2 in 6.55 seconds.
Processed cutout 3 in 10.09 seconds.
Processed cutout 4 in 5.57 seconds.
Processed cutout 5 in 7.51 seconds.
Processed cutout 6 in 7.75 seconds.


KeyboardInterrupt: 

### Parallel

In [14]:
results_table_parallel = results.to_table()
results_table_parallel["cutout_index"] = range(1, len(results_table_parallel) + 1)
results_table_parallel["central_wavelength"] = np.full(
    len(results_table_parallel), np.nan
)
results_table_parallel["version"] = np.full(
    len(results_table_parallel), "", dtype="U32"
)
results_table_parallel["hdus"] = np.full(len(results_table_parallel), None)

t1 = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    futures = [
        executor.submit(process_cutout_with_error_handling, row, coord, False)
        for row in results_table_parallel
    ]
    concurrent.futures.wait(futures)
print(
    f"Time to create cutouts in parallel mode: {(time.time() - t1) / 60:2.2f} minutes."
)

# Drop rows that failed to download.
failed_cutouts_parallel = results_table_parallel[results_table_parallel["hdus"] is None]
print("Failed to get the following cutouts:\n", failed_cutouts_parallel["uri"])
results_table_parallel = results_table_parallel[
    results_table_parallel["hdus"] is not None
]

Time to create cutouts in parallel mode: 1.48 minutes.
Failed to get the following cutouts:
 https://irsa.ipac.caltech.edu/ibe/data/spherex/qr2/level2/2025W19_2B/l2b-v20-2025-247/2/level2_2025W19_2B_0346_1D2_spx_l2b-v20-2025-247.fits?center=210.80227,54.34895d&size=0.1


In [18]:
results_table_parallel

uri,time_bounds_lower,cutout_index,central_wavelength,version,hdus
,d,,,,
object,float64,int64,float64,str32,object
"https://irsa.ipac.caltech.edu/ibe/data/spherex/qr2/level2/2025W19_2B/l2b-v20-2025-247/2/level2_2025W19_2B_0346_2D2_spx_l2b-v20-2025-247.fits?center=210.80227,54.34895d&size=0.1",60805.713040,2,1.542889310216153,6.4+psffix1,"[<astropy.io.fits.hdu.image.ImageHDU object at 0x7fec3f2f1bd0>, <astropy.io.fits.hdu.image.ImageHDU object at 0x7fec3f2f3c50>, <astropy.io.fits.hdu.image.ImageHDU object at 0x7fec3f42a710>, <astropy.io.fits.hdu.image.ImageHDU object at 0x7fec3f428190>, <astropy.io.fits.hdu.image.ImageHDU object at 0x7fec3f4287d0>, <astropy.io.fits.hdu.table.BinTableHDU object at 0x7fec3f0af150>]"


## Summary table

In [15]:
# Create a summary table HDU with renamed columns
cols = fits.ColDefs(
    [
        fits.Column(
            name="cutout_index",
            format="J",
            array=results_table_parallel["cutout_index"],
            unit="",
        ),
        fits.Column(
            name="observation_date",
            format="D",
            array=results_table_parallel["time_bounds_lower"],
            unit="d",
        ),
        fits.Column(
            name="central_wavelength",
            format="D",
            array=results_table_parallel["central_wavelength"],
            unit="um",
        ),
        fits.Column(
            name="version",
            format="A32",
            array=results_table_parallel["version"],
            unit="",
        ),
        fits.Column(
            name="access_url",
            format="A200",
            array=results_table_parallel["uri"],
            unit="",
        ),
    ]
)
table_hdu = fits.BinTableHDU.from_columns(cols)
table_hdu.header["EXTNAME"] = "CUTOUT_INFO"

IndexError: tuple index out of range